In [34]:
import pandas as pd
import numpy as np
import tensorflow as tf
from keras.models import Model
from keras.layers import Input, Dense, LSTM, Layer, Concatenate, Subtract
from keras.src.losses import BinaryCrossentropy
from keras.optimizers import Adam
from keras.activations import relu
from keras.initializers import RandomUniform
from keras.metrics import binary_accuracy

In [5]:
columns = [
    "latitude", "longitude", "unused", "altitude",
    "datetime_float", "date", "time"
]

# Read the file
df = pd.read_csv(
    "../data/Geolife Trajectories/000/Trajectory/20081023025304.plt",
    skiprows=6,          # skip header lines
    names=columns,
    header=None
)

# Drop the unused column if not needed
df = df.drop(columns=["unused"])

# Optionally: Combine date and time into a single datetime column
df["datetime"] = pd.to_datetime(df["date"] + " " + df["time"])

In [6]:
df.head(n=100)

,latitude,longitude,altitude,datetime_float,date,time,datetime
0,39.984702,116.318417,492,39744.120185,2008-10-23,02:53:04,2008-10-23 02:53:04
1,39.984683,116.318450,492,39744.120255,2008-10-23,02:53:10,2008-10-23 02:53:10
2,39.984686,116.318417,492,39744.120313,2008-10-23,02:53:15,2008-10-23 02:53:15
3,39.984688,116.318385,492,39744.120370,2008-10-23,02:53:20,2008-10-23 02:53:20
4,39.984655,116.318263,492,39744.120428,2008-10-23,02:53:25,2008-10-23 02:53:25
...,...,...,...,...,...,...,...
95,39.984380,116.302754,240,39744.125694,2008-10-23,03:01:00,2008-10-23 03:01:00
96,39.984346,116.302601,239,39744.125752,2008-10-23,03:01:05,2008-10-23 03:01:05
97,39.984303,116.302457,237,39744.125810,2008-10-23,03:01:10,2008-10-23 03:01:10
98,39.984344,116.302271,233,39744.125868,2008-10-23,03:01:15,2008-10-23 03:01:15


In [7]:
df.describe()

,latitude,longitude,altitude,datetime_float,datetime
count,908.000000,908.000000,908.000000,908.000000,908
mean,39.998040,116.315561,398.093612,39744.289663,2008-10-23 06:57:06.887665152
min,39.983276,116.285446,-407.000000,39744.120185,2008-10-23 02:53:04
25%,39.985143,116.310045,130.000000,39744.177034,2008-10-23 04:14:55.750000128
50%,39.999658,116.319314,153.000000,39744.190041,2008-10-23 04:33:39.500000
75%,40.008201,116.321480,206.000000,39744.422885,2008-10-23 10:08:57.249999872
max,40.009428,116.324886,7584.000000,39744.466111,2008-10-23 11:11:12
std,0.010490,0.007881,1190.964197,0.132342,NaN


In [8]:
LATITUDE_MIN = -90.0
LATITUDE_MAX = 90.0
LONGITUDE_MIN = -180.0
LONGITUDE_MAX = 180.0

df["latitude"] = (df["latitude"] - LATITUDE_MIN) / (LATITUDE_MAX - LATITUDE_MIN)
df["longitude"] = (df["longitude"] - LONGITUDE_MIN) / (LONGITUDE_MAX - LONGITUDE_MIN)

df.describe()

,latitude,longitude,altitude,datetime_float,datetime
count,908.000000,908.000000,908.000000,908.000000,908
mean,0.722211,0.823099,398.093612,39744.289663,2008-10-23 06:57:06.887665152
min,0.722129,0.823015,-407.000000,39744.120185,2008-10-23 02:53:04
25%,0.722140,0.823083,130.000000,39744.177034,2008-10-23 04:14:55.750000128
50%,0.722220,0.823109,153.000000,39744.190041,2008-10-23 04:33:39.500000
75%,0.722268,0.823115,206.000000,39744.422885,2008-10-23 10:08:57.249999872
max,0.722275,0.823125,7584.000000,39744.466111,2008-10-23 11:11:12
std,0.000058,0.000022,1190.964197,0.132342,NaN


In [24]:
class Time2Vec(Layer):

    def __init__(self, output_dim=2, **kwargs):
        if output_dim < 2:
            raise ValueError("output_dim must be at least 2")
        self.output_dim = output_dim
        self.p = None
        self.w = None
        self.P = None
        self.W = None
        super(Time2Vec, self).__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(
            name="W",
            shape=(input_shape[-1], self.output_dim - 1),
            initializer=RandomUniform(),
            trainable=True
        )

        self.P = self.add_weight(
            name="P",
            shape=(self.output_dim - 1,),
            initializer=RandomUniform(),
            trainable=True
        )

        self.w = self.add_weight(
            name="w",
            shape=(input_shape[-1], 1),
            initializer=RandomUniform(),
            trainable=True
        )

        self.p = self.add_weight(
            name="p",
            shape=(1,),
            initializer=RandomUniform(),
            trainable=True
        )

    def call(self, inputs):
        linear_part = tf.add(tf.matmul(inputs, self.w), self.p)
        periodic_part = tf.sin(tf.add(tf.matmul(inputs, self.W), self.P))
        return tf.concat([linear_part, periodic_part], axis=-1)

In [32]:
# Define the model architecture

## Define memory encoder
location_memory_input = Input(shape=(None, 2))
time_memory_input = Input(shape=(None, 1))
time2vec_memory_input = Time2Vec(output_dim=8)(time_memory_input)
memory_input = Concatenate()([location_memory_input, time2vec_memory_input])
memory_encoder = LSTM(32, return_sequences=False)(memory_input)

## Define query encoder
location_query_input = Input(shape=(None, 2))
time_query_input = Input(shape=(None, 1))
time2vec_query_input = Time2Vec(output_dim=8)(time_query_input)
query_input = Concatenate()([location_query_input, time2vec_query_input])
query_encoder = LSTM(32, return_sequences=False)(query_input)

## Define output layer
subtract_layer = Subtract()([memory_encoder, query_encoder])
hidden_layer = Dense(32, activation=relu)(subtract_layer)
output_layer = Dense(1, activation="sigmoid")(hidden_layer)

In [33]:
model = Model(inputs=[location_memory_input, time_memory_input], outputs=output_layer)
model.compile(optimizer=Adam(), loss=BinaryCrossentropy(), metrics=[binary_accuracy])
model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_5       │ (None, None, 1)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_7       │ (None, None, 1)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_4       │ (None, None, 2)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time2_vec_2         │ (None, None, 8)   │         16 │ input_layer_5[0]… │
│ (Time2Vec)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_6       │ (None, None, 2)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time2_vec_3         │ (None, None, 8)   │         16 │ input_layer_7[0]… │
│ (Time2Vec)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_6       │ (None, None, 10)  │          0 │ input_layer_4[0]… │
│ (Concatenate)       │                   │            │ time2_vec_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_7       │ (None, None, 10)  │          0 │ input_layer_6[0]… │
│ (Concatenate)       │                   │            │ time2_vec_3[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_6 (LSTM)       │ (None, 32)        │      5,504 │ concatenate_6[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_7 (LSTM)       │ (None, 32)        │      5,504 │ concatenate_7[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ subtract_3          │ (None, 32)        │          0 │ lstm_6[0][0],     │
│ (Subtract)          │                   │            │ lstm_7[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 32)        │      1,056 │ subtract_3[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 1)         │         33 │ dense_6[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 12,129 (47.38 KB)

 Trainable params: 12,129 (47.38 KB)

 Non-trainable params: 0 (0.00 B)